In [1]:
import os
from pathlib import Path
import pandas as pd

os.chdir(r'C:\Users\HP\Documents\Economy\economia_c')
print("Directorio:", Path.cwd())

RUTA = Path.cwd() / "data" / "raw" / "provinciales_serie_remuneraciones_mensual_4dig_7.xlsx"

xls = pd.ExcelFile(RUTA)
print("\nHojas:", xls.sheet_names)

df = pd.read_excel(RUTA, sheet_name="La Rioja", header=None)
print("\nShape:", df.shape)
print("\nPrimeras 8 filas:")
print(df.iloc[:8, :8].to_string())

Directorio: C:\Users\HP\Documents\Economy\economia_c

Hojas: ['Caratula', 'Indice', 'GBA', 'CABA', 'Resto de Buenos Aires', 'Catamarca', 'Córdoba', 'Corrientes', 'Chaco', 'Chubut', 'Entre Ríos', 'Formosa', 'Jujuy', 'La Pampa', 'La Rioja', 'Mendoza', 'Misiones', 'Neuquén', 'Río Negro', 'Salta', 'San Juan', 'San Luis', 'Santa Cruz', 'Santa Fe', 'Santiago del Estero', 'Tucumán', 'Tierra del Fuego', 'Metodología']

Shape: (313, 378)

Primeras 8 filas:
    0                                                                                                                                    1            2                    3                    4                    5                    6                    7
0 NaN                                                                                                                                  NaN          NaN                  NaN                  NaN                  NaN                  NaN                  NaN
1 NaN                             

In [2]:
# Header con fechas (fila 6)
header = df.iloc[6, :].tolist()

# Buscar columnas de dic-2024 y dic-2025
col_dic2024 = None
col_dic2025 = None

for i, h in enumerate(header):
    if pd.notna(h):
        fecha = pd.to_datetime(h, errors="coerce")
        if pd.notna(fecha):
            if fecha.year == 2024 and fecha.month == 12:
                col_dic2024 = i
            if fecha.year == 2025 and fecha.month == 12:
                col_dic2025 = i

print(f"Columna dic-2024: {col_dic2024}")
print(f"Columna dic-2025: {col_dic2025}")
print()

# Buscar códigos del núcleo informático
CIIU_NUCLEO = [7220, 7250, 7290]
CIIU_PROXY = [7421, 7499]

print("=" * 70)
print("CÓDIGOS DEL NÚCLEO INFORMÁTICO")
print("=" * 70)
for idx in range(8, len(df)):
    rama = df.iloc[idx, 0]
    try:
        rama_int = int(rama) if pd.notna(rama) else None
    except (ValueError, TypeError):
        continue
    if rama_int in CIIU_NUCLEO + CIIU_PROXY:
        nombre = str(df.iloc[idx, 1])[:55]
        v24 = df.iloc[idx, col_dic2024] if col_dic2024 else None
        v25 = df.iloc[idx, col_dic2025] if col_dic2025 else None
        print(f"  {rama_int} — {nombre}")
        print(f"      dic-2024: {v24}")
        print(f"      dic-2025: {v25}")
        print()

Columna dic-2024: 362
Columna dic-2025: 374

CÓDIGOS DEL NÚCLEO INFORMÁTICO


In [3]:
# Ver cómo vienen los códigos en la columna 0
print("Tipos y valores de la columna 0 (primeros 30):")
print()
for idx in range(8, 40):
    rama = df.iloc[idx, 0]
    desc = str(df.iloc[idx, 1])[:40]
    print(f"  Fila {idx}: tipo={type(rama).__name__}, valor={repr(rama)}, desc={desc}")

Tipos y valores de la columna 0 (primeros 30):

  Fila 8: tipo=float64, valor=np.float64(nan), desc=111
  Fila 9: tipo=float64, valor=np.float64(nan), desc=112
  Fila 10: tipo=float64, valor=np.float64(nan), desc=113
  Fila 11: tipo=float64, valor=np.float64(nan), desc=114
  Fila 12: tipo=float64, valor=np.float64(nan), desc=115
  Fila 13: tipo=float64, valor=np.float64(nan), desc=121
  Fila 14: tipo=float64, valor=np.float64(nan), desc=122
  Fila 15: tipo=float64, valor=np.float64(nan), desc=141
  Fila 16: tipo=float64, valor=np.float64(nan), desc=142
  Fila 17: tipo=float64, valor=np.float64(nan), desc=150
  Fila 18: tipo=float64, valor=np.float64(nan), desc=201
  Fila 19: tipo=float64, valor=np.float64(nan), desc=202
  Fila 20: tipo=float64, valor=np.float64(nan), desc=203
  Fila 21: tipo=float64, valor=np.float64(nan), desc=501
  Fila 22: tipo=float64, valor=np.float64(nan), desc=502
  Fila 23: tipo=float64, valor=np.float64(nan), desc=503
  Fila 24: tipo=float64, valor=np.float64(

In [4]:
CIIU_BUSCAR = [7220, 7250, 7290, 7421, 7499]

print("=" * 70)
print("SALARIOS DIC-2024 vs DIC-2025")
print("=" * 70)
print()

# Header de fechas: fila 6
header = df.iloc[6, :].tolist()

col_dic2024 = None
col_dic2025 = None
for i, h in enumerate(header):
    if pd.notna(h):
        fecha = pd.to_datetime(h, errors="coerce")
        if pd.notna(fecha):
            if fecha.year == 2024 and fecha.month == 12:
                col_dic2024 = i
            if fecha.year == 2025 and fecha.month == 12:
                col_dic2025 = i

print(f"Col dic-2024: {col_dic2024}, Col dic-2025: {col_dic2025}")
print()

# Buscar en columna 1
for idx in range(8, len(df)):
    rama = df.iloc[idx, 1]  # ← columna 1, no 0
    if pd.isna(rama):
        continue
    
    rama_str = str(rama).strip().replace(".0", "")
    
    for codigo in CIIU_BUSCAR:
        if rama_str == str(codigo):
            nombre = str(df.iloc[idx, 2])[:55]  # descripción en col 2
            v24 = df.iloc[idx, col_dic2024]
            v25 = df.iloc[idx, col_dic2025]
            print(f"  {codigo} — {nombre}")
            print(f"      dic-2024: {v24}")
            print(f"      dic-2025: {v25}")
            print()
            break

SALARIOS DIC-2024 vs DIC-2025

Col dic-2024: 362, Col dic-2025: 374

  7220 — Servicios de consultores en informática y suministros d
      dic-2024: 1976383.5083333335
      dic-2025: 2786240.2883333336

  7250 — Mantenimiento y reparación de maquinaria de oficina, co
      dic-2024: 1233256.9692307692
      dic-2025: 0

  7290 — Actividades de informática n.c.p.
      dic-2024: 3347794.114074075
      dic-2025: 4220277.553333334

  7421 — Servicios de arquitectura e ingeniería y servicios cone
      dic-2024: 1099990.981849711
      dic-2025: 1566575.0660335205

  7499 — Servicios empresariales n.c.p.
      dic-2024: 1336469.3323320171
      dic-2025: 1534016.4095419853



In [5]:
# IPC de Argentina (INDEC)
IPC_2025 = 0.315   # 31,5% inflación anual dic-2024 a dic-2025

# Salarios dic-2024 y dic-2025 (extraídos del archivo)
datos = {
    7220: {"nombre": "Consultores informática", "dic24": 1976383.51, "dic25": 2786240.29},
    7250: {"nombre": "Mantenimiento",           "dic24": 1233256.97, "dic25": 0},
    7290: {"nombre": "Informática n.c.p.",      "dic24": 3347794.11, "dic25": 4220277.55},
    7421: {"nombre": "Arquitectura/ingeniería", "dic24": 1099990.98, "dic25": 1566575.07},
    7499: {"nombre": "Servicios empresariales", "dic24": 1336469.33, "dic25": 1534016.41},
}

print("=" * 85)
print("VARIACIÓN INTERANUAL REAL DE SALARIOS (dic-2024 vs dic-2025)")
print(f"IPC 2025: +{IPC_2025*100:.1f}%")
print("=" * 85)
print()
print(f"{'CIIU':<6} {'Rama':<28} {'dic-2024':>13} {'dic-2025':>13} {'Var. real':>10}")
print("-" * 85)

for ciiu, d in datos.items():
    if d["dic25"] == 0:
        print(f"{ciiu:<6} {d['nombre']:<28} {d['dic24']:>13,.0f} {'s/d':>13} {'s/d':>10}")
        continue
    
    # Salario dic-2024 ajustado a pesos de dic-2025
    ajustado = d["dic24"] * (1 + IPC_2025)
    
    # Variación real
    var_real = (d["dic25"] / ajustado - 1) * 100
    
    print(f"{ciiu:<6} {d['nombre']:<28} {d['dic24']:>13,.0f} {d['dic25']:>13,.0f} {var_real:>+9.2f}%")

print("-" * 85)

VARIACIÓN INTERANUAL REAL DE SALARIOS (dic-2024 vs dic-2025)
IPC 2025: +31.5%

CIIU   Rama                              dic-2024      dic-2025  Var. real
-------------------------------------------------------------------------------------
7220   Consultores informática          1,976,384     2,786,240     +7.21%
7250   Mantenimiento                    1,233,257           s/d        s/d
7290   Informática n.c.p.               3,347,794     4,220,278     -4.14%
7421   Arquitectura/ingeniería          1,099,991     1,566,575     +8.30%
7499   Servicios empresariales          1,336,469     1,534,016    -12.71%
-------------------------------------------------------------------------------------
